**Strategy 1 | Long + Short | SMA Cross Regime | EMA 12 Stop Loss | Time Exit**

Changes vs. v2.4:
- Label: three-class (SELL=0, FLAT=1, BUY=2) — v4.1 SMA Cross regime
- Regime: Bull = SMA50 > SMA200 (symmetric for Bear)
- Stop Loss: EMA 12 dynamic (long: close < EMA12 | short: close > EMA12)
- Exit: Time Exit after N=5 bars
- Position Sizing: configurable via POSITION_SIZE (currently 100%)
- Transaction Costs: directly tracked per trade

In [43]:
import pandas as pd
import pandas_ta as ta
import lightgbm as lgb
import numpy as np

# Loading the Data

In [44]:
df = pd.read_csv('btc_1hour_cleaned.csv')

## Setting the correct formats and cleaning Dataframe

In [45]:
df['Open time'] = pd.to_datetime(df['Open time'])
df['Close time'] = pd.to_datetime(df['Close time'])

In [46]:
df = df.set_index('Open time')

# Feature Engineering

## Functions for features: Strategy 1 + regime

In [47]:
# ROC
def compute_roc(df):
    df['roc_10'] = df['Close'].pct_change(periods=10)
    df['roc_21'] = df['Close'].pct_change(periods=21)
    return df

# MACD Histogram
def compute_macd(df):
    macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
    df['macd_histogram'] = macd['MACDh_12_26_9']
    return df

# ADX
def compute_adx(df):
    df['adx'] = ta.adx(df['High'], df['Low'], df['Close'], length=14)['ADX_14']
    return df

# EMA 12 — used as dynamic stop loss in execution layer
def compute_ema12(df):
    df['ema_12'] = df['Close'].ewm(span=12, adjust=False).mean()
    return df

# SMA 50 and SMA 200 is purely used for regime detection
def compute_sma(df):
    df['sma_50']  = df['Close'].rolling(50).mean()
    df['sma_200'] = df['Close'].rolling(200).mean()
    return df

# ATR 14
def compute_atr(df):
    df['atr_14'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    return df

# NATR
def compute_natr(df):
    df['natr'] = ta.natr(df['High'], df['Low'], df['Close'], length=14)
    return df

## Pipeline

In [48]:
def feature_pipeline(df):
    steps = [
        # model features (go to X)
        compute_roc,
        compute_macd,
        compute_adx,
        compute_atr,
        compute_natr,
        # execution layer only (does not go into X)
        compute_sma,
        compute_ema12,
    ]
    for step in steps:
        df = step(df)
    return df

In [49]:
df = feature_pipeline(df)

# Creating labels (y)

In [50]:
# Lookahead window - change this value to test different horizons
N = 5

# Step 1 - Forward return
df['forward_return'] = df['Close'].pct_change(periods=N).shift(-N)

In [51]:
# Step 2 - Three-Class Label (v4.1 SMA Cross Regime — symmetric)
# BUY=2:  positive return AND Bull
# SELL=0: negative return AND Bear
# FLAT=1: everything else

# v4.1: SMA Cross only — no Close condition required
bull_cross = (df['sma_50'] > df['sma_200'])
bear_cross = (df['sma_50'] < df['sma_200'])

conditions = [
    (df['forward_return'] > 0) & bull_cross,  # BUY  = 2
    (df['forward_return'] < 0) & bear_cross,  # SELL = 0
]

df['label'] = np.select(conditions, [2, 0], default=1)  # FLAT = 1

print(f"SELL (0): {(df['label'] == 0).mean():.1%}")
print(f"FLAT (1): {(df['label'] == 1).mean():.1%}")
print(f"BUY  (2): {(df['label'] == 2).mean():.1%}")

SELL (0): 23.6%
FLAT (1): 49.9%
BUY  (2): 26.5%


# Creating the lags

In [52]:
# Number of lags to create
N_lags = 5

feature_cols = ['Volume','roc_10', 'roc_21', 'macd_histogram', 'adx']

# Create lags for all feature columns
for col in feature_cols:
    for lag in range(1, N_lags + 1):
        df[f'{col}_lag{lag}'] = df[col].shift(lag)

# Drop NaN rows created by the lags
df = df.dropna(subset=feature_cols + [f'{col}_lag{i}' for col in feature_cols for i in range(1, N_lags + 1)] + ['label'])

# Creating the Feature Matrix and Y

In [53]:
X = df[['Volume','roc_10', 'roc_21', 'macd_histogram', 'adx',
                     'Volume_lag1', 'Volume_lag2', 'Volume_lag3', 'Volume_lag4', 'Volume_lag5',
                     'roc_10_lag1', 'roc_10_lag2', 'roc_10_lag3', 'roc_10_lag4', 'roc_10_lag5',
                     'roc_21_lag1', 'roc_21_lag2', 'roc_21_lag3', 'roc_21_lag4', 'roc_21_lag5',
                     'macd_histogram_lag1', 'macd_histogram_lag2', 'macd_histogram_lag3', 'macd_histogram_lag4', 'macd_histogram_lag5',
                     'adx_lag1', 'adx_lag2', 'adx_lag3', 'adx_lag4', 'adx_lag5']]

In [54]:
y = df['label']

# Training the Model

## Creating the Train / Test split according to user Input

In [55]:
# Hardcoding the date entered by user, to be set up dynamically in next versions
CUTOFF_DATE = '2025-01-01'

# Creating the training set for X and y
X_train = X[X.index < CUTOFF_DATE]
y_train = y[y.index < CUTOFF_DATE]

# Creating the testing set for X and y
X_predict = X[X.index >= CUTOFF_DATE]
y_predict = y[y.index >= CUTOFF_DATE]  # kept for evaluation only

print(f'Training on {len(X_train)} bars up to {CUTOFF_DATE}')
print(f'Predicting on {len(X_predict)} bars from {CUTOFF_DATE} onwards')

Training on 61208 bars up to 2025-01-01
Predicting on 10342 bars from 2025-01-01 onwards


## Training the model with LightGBM

In [ ]:
# Multiclass params — is_unbalance=True auto-weights all 3 classes
params = {
    'objective':        'multiclass',
    'metric':           'multi_logloss',
    'num_class':        3,
    'boosting_type':    'gbdt',
    'learning_rate':    0.05,
    'num_leaves':       31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'is_unbalance':     True,
   # 'seed':             42, #no random state for production
    'verbose':          -1
}

In [57]:
# Packaging the training data into LightGBM's required format
train_set_lgb = lgb.Dataset(X_train, label=y_train)

# Training the model
final_model= lgb.train(
    params= params,
    train_set= train_set_lgb,
    num_boost_round= 300,
    callbacks= [lgb.log_evaluation(period=-1)]
)

# Generate predictions on post-cutoff data
pred_proba_final = final_model.predict(X_predict)

## Building the signals DataFrame - this is the input to the execution filter

In [58]:
signals_df = df.loc[X_predict.index, ['Close', 'sma_50', 'sma_200', 'adx', 'atr_14', 'ema_12']].copy()

# pred_proba_final is now shape (n_bars, 3) — one column per class
signals_df['proba_sell'] = pred_proba_final[:, 0]
signals_df['proba_flat'] = pred_proba_final[:, 1]
signals_df['proba_buy']  = pred_proba_final[:, 2]
signals_df['true_label'] = y_predict.values

# Execution Filter + P&L Tracking

Purpose: Convert raw model probabilities into trade actions bar by bar
- Long: enter when Bull regime confirmed (SMA50 > SMA200) AND P(BUY) >= threshold
- Short: enter when Bear regime confirmed (SMA50 < SMA200) AND P(SELL) >= threshold
- Exit 1: EMA 12 stop loss (long: close < EMA12 | short: close > EMA12)
- Exit 2: Time exit after N=5 bars

In [59]:
# Setting the parameters of the execution strategy
CONFIDENCE_THRESHOLD = 0.55
INITIAL_CAPITAL = 1000.0

COST_PCT = 0.001  # 0.1% transaction cost per side
POSITION_SIZE = 1.0  # fraction of capital per trade — change here

In [60]:
# Setting the loop that executes the trades

# State variables
position        = 'flat'
entry_price     = 0.0
trade_pos_size  = 0.0
capital         = INITIAL_CAPITAL
N_HOLD          = 5
bars_in_trade   = 0

trade_log = []

for timestamp, row in signals_df.iterrows():
    close      = row['Close']
    sma_50     = row['sma_50']
    sma_200    = row['sma_200']
    adx        = row['adx']
    atr        = row['atr_14']
    proba_buy  = row['proba_buy']
    proba_sell = row['proba_sell']

    if pd.isna(sma_50) or pd.isna(sma_200) or pd.isna(row['ema_12']):
        continue

    # Regime — SMA Cross only
    bull = (sma_50 > sma_200)
    bear = (sma_50 < sma_200)

    # ── LONG MANAGEMENT ───────────────────────────────────────────────────
    if position == 'long':

        # EXIT 1: Stop Loss — close below EMA 12
        if close < row['ema_12']:
            pnl  = (row['ema_12'] - entry_price) * trade_pos_size
            cost = row['ema_12'] * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp, 'action': 'stop_loss_long',
                'close': row['ema_12'], 'pnl': pnl, 'cost': cost, 'capital': capital
            })
            position, entry_price, trade_pos_size, bars_in_trade = 'flat', 0.0, 0.0, 0
            continue

        # EXIT 2: Time Exit
        bars_in_trade += 1
        if bars_in_trade >= N_HOLD:
            pnl  = (close - entry_price) * trade_pos_size
            cost = close * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp, 'action': 'close_long_time',
                'close': close, 'pnl': pnl, 'cost': cost, 'capital': capital
            })
            position, entry_price, trade_pos_size, bars_in_trade = 'flat', 0.0, 0.0, 0
            continue

    # ── SHORT MANAGEMENT ──────────────────────────────────────────────────
    elif position == 'short':

        # EXIT 1: Stop Loss — close above EMA 12
        if close > row['ema_12']:
            pnl  = (entry_price - row['ema_12']) * trade_pos_size
            cost = row['ema_12'] * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp, 'action': 'stop_loss_short',
                'close': row['ema_12'], 'pnl': pnl, 'cost': cost, 'capital': capital
            })
            position, entry_price, trade_pos_size, bars_in_trade = 'flat', 0.0, 0.0, 0
            continue

        # EXIT 2: Time Exit
        bars_in_trade += 1
        if bars_in_trade >= N_HOLD:
            pnl  = (entry_price - close) * trade_pos_size
            cost = close * trade_pos_size * COST_PCT
            capital += pnl - cost
            trade_log.append({
                'timestamp': timestamp, 'action': 'close_short_time',
                'close': close, 'pnl': pnl, 'cost': cost, 'capital': capital
            })
            position, entry_price, trade_pos_size, bars_in_trade = 'flat', 0.0, 0.0, 0
            continue

    # ── ENTRY BLOCK ───────────────────────────────────────────────────────
    else:
        if bull and proba_buy >= CONFIDENCE_THRESHOLD:
            trade_pos_size = (capital * POSITION_SIZE) / close
            cost = close * trade_pos_size * COST_PCT
            capital -= cost
            position, entry_price, bars_in_trade = 'long', close, 0
            trade_log.append({
                'timestamp': timestamp, 'action': 'enter_long',
                'close': close, 'pnl': 0, 'cost': cost, 'capital': capital
            })

        elif bear and proba_sell >= CONFIDENCE_THRESHOLD:
            trade_pos_size = (capital * POSITION_SIZE) / close
            cost = close * trade_pos_size * COST_PCT
            capital -= cost
            position, entry_price, bars_in_trade = 'short', close, 0
            trade_log.append({
                'timestamp': timestamp, 'action': 'enter_short',
                'close': close, 'pnl': 0, 'cost': cost, 'capital': capital
            })

## Performance Summary

In [61]:
import numpy as np

# ================================================================
# FULL PERFORMANCE EVALUATION
# ================================================================

# --- Setup ---
trade_df       = pd.DataFrame(trade_log)
entries        = trade_df[trade_df['action'].isin(['enter_long', 'enter_short'])]

exits          = trade_df[~trade_df['action'].isin(['enter_long', 'enter_short'])]
winning_trades = exits[exits['pnl'] > 0]
losing_trades  = exits[exits['pnl'] < 0]
total_closed   = len(exits)
win_rate       = len(winning_trades) / total_closed * 100 if total_closed > 0 else 0
loss_rate      = len(losing_trades)  / total_closed * 100 if total_closed > 0 else 0

# --- Risk Metrics ---
trade_df_sorted = trade_df.sort_values('timestamp')
trade_df_sorted['capital_return'] = trade_df_sorted['capital'].pct_change()
mean_return      = trade_df_sorted['capital_return'].mean()
std_return       = trade_df_sorted['capital_return'].std()

# --- Backtest Window ---
backtest_start   = trade_df_sorted['timestamp'].iloc[0].strftime('%d %b %Y')
backtest_end     = trade_df_sorted['timestamp'].iloc[-1].strftime('%d %b %Y')
days_in_backtest = (trade_df_sorted['timestamp'].iloc[-1] - trade_df_sorted['timestamp'].iloc[0]).days
annualised_return = ((capital / INITIAL_CAPITAL) ** (365 / days_in_backtest) - 1) * 100

# ── Corrected Sharpe Ratio ─────────────────────────────────────────────────
# Risk-free rate: 4.25% annualised — approximates average US Federal Funds
# effective rate over the Jan 2024 – Mar 2026 evaluation window
# Annualisation: based on actual average trade frequency (not assumed hourly)
# This replaces the previous sqrt(8760) approach which was overstated

RISK_FREE_RATE  = 0.0425   # 4.25% annualised

# Compute average holding period in hours from actual trade timestamps
entry_times = trade_df[trade_df['action'] == 'enter_long']['timestamp'].values
exit_times  = trade_df[trade_df['action'] != 'enter_long']['timestamp'].values

if len(entry_times) == len(exit_times) and len(entry_times) > 0:
    holding_hours   = np.mean([
        (pd.Timestamp(ex) - pd.Timestamp(en)).total_seconds() / 3600
        for en, ex in zip(entry_times, exit_times)
    ])
    trades_per_year = 8760 / holding_hours
else:
    # Fallback if entries/exits are unbalanced — use trade count over period
    trades_per_year = total_closed / (days_in_backtest / 365)
    holding_hours   = 8760 / trades_per_year

# Risk-free rate per trade observation
rf_per_trade  = RISK_FREE_RATE / trades_per_year

# Sharpe — excess return per unit of risk, annualised at trade frequency
sharpe        = ((mean_return - rf_per_trade) / std_return) * np.sqrt(trades_per_year)

# ── End Sharpe ─────────────────────────────────────────────────────────────

trade_df_sorted['cummax']   = trade_df_sorted['capital'].cummax()
trade_df_sorted['drawdown'] = (trade_df_sorted['capital'] - trade_df_sorted['cummax']) / trade_df_sorted['cummax']
max_drawdown     = trade_df_sorted['drawdown'].min() * 100
gross_profit     = winning_trades['pnl'].sum()
gross_loss       = losing_trades['pnl'].abs().sum()
profit_factor    = gross_profit / gross_loss if gross_loss > 0 else 0
days_in_backtest = (trade_df_sorted['timestamp'].iloc[-1] - trade_df_sorted['timestamp'].iloc[0]).days
annualised_return = ((capital / INITIAL_CAPITAL) ** (365 / days_in_backtest) - 1) * 100

# --- Buy and Hold Benchmark ---
buy_date    = CUTOFF_DATE
buy_price   = df.loc[buy_date:, 'Close'].iloc[0]
final_price = df['Close'].iloc[-1]
btc_units   = INITIAL_CAPITAL / buy_price
bnh_value   = btc_units * final_price
bnh_return  = ((bnh_value - INITIAL_CAPITAL) / INITIAL_CAPITAL) * 100

# --- Transaction Costs ---
total_transaction_costs = trade_df['cost'].sum()

# --- Position Sizing Analysis ---
# Reconstruct actual position size per trade from pnl and price movement
entry_trades = trade_df[trade_df['action'].isin(['enter_long', 'enter_short'])].copy()

# Average position value = x% of capital at entry
# We don't track capital at each entry in trade_log, so use average capital
avg_capital     = (INITIAL_CAPITAL + capital) / 2  # average of start and end
avg_pos_usd     = avg_capital * POSITION_SIZE
avg_pos_btc     = avg_pos_usd / entry_trades['close'].mean()
avg_risk_dollar = avg_pos_usd


# --- Print All Results ---
print(f'╔══════════════════════════════════════╗')
print(f'║         BACKTEST RESULTS             ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ CAPITAL                              ║')
print(f'║  Period: {backtest_start} → {backtest_end}   ║')
print(f'║  Initial:         ${INITIAL_CAPITAL:>10,.2f}        ║')
print(f'║  Final:           ${capital:>10,.2f}        ║')
print(f'║  Total Return:    {((capital - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100):>9.2f}%         ║')
print(f'║  Annualised:      {annualised_return:>9.2f}%         ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ TRADE STATISTICS                     ║')
print(f'║  Total Trades:    {len(entries):>10}         ║')
print(f'║  Winning Trades:  {len(winning_trades):>10}         ║')
print(f'║  Losing Trades:   {len(losing_trades):>10}         ║')
print(f'║  Win Rate:        {win_rate:>9.1f}%         ║')
print(f'║  Loss Rate:       {loss_rate:>9.1f}%         ║')
print(f'║  Avg Win PnL:     ${winning_trades["pnl"].mean():>10.2f}        ║')
print(f'║  Avg Loss PnL:    ${losing_trades["pnl"].mean():>10.2f}        ║')
print(f'║  Transaction Costs: ${total_transaction_costs:>8.2f}        ║')
print(f'║  Avg Risk per Trade:${avg_risk_dollar:>9.2f}       ║')
print(f'║  Avg Position (BTC):{avg_pos_btc:>9.6f}        ║')
print(f'║  Avg Position ($):  ${avg_pos_usd:>9.2f}       ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ RISK METRICS                         ║')
print(f'║  Sharpe Ratio:    {sharpe:>10.2f}         ║')
print(f'║  Max Drawdown:    {max_drawdown:>9.2f}%         ║')
print(f'║  Profit Factor:   {profit_factor:>10.2f}         ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ BUY & HOLD BENCHMARK                 ║')
print(f'║  Buy Price:       ${buy_price:>10,.2f}        ║')
print(f'║  Final Price:     ${final_price:>10,.2f}        ║')
print(f'║  Final Value:     ${bnh_value:>10,.2f}        ║')
print(f'║  B&H Return:      {bnh_return:>9.2f}%         ║')
print(f'║  Strategy Return: {((capital - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100):>9.2f}%         ║')
print(f'╠══════════════════════════════════════╣')
print(f'║ ACTION BREAKDOWN                     ║')
for action, count in trade_df['action'].value_counts().items():
    print(f'║  {action:<20} {count:>6}         ║')
print(f'╚══════════════════════════════════════╝')

╔══════════════════════════════════════╗
║         BACKTEST RESULTS             ║
╠══════════════════════════════════════╣
║ CAPITAL                              ║
║  Initial:         $  1,000.00        ║
║  Final:           $  1,791.60        ║
║  Total Return:        79.16%         ║
║  Annualised:          64.81%         ║
╠══════════════════════════════════════╣
║ TRADE STATISTICS                     ║
║  Total Trades:           266         ║
║  Winning Trades:         224         ║
║  Losing Trades:           42         ║
║  Win Rate:             84.2%         ║
║  Loss Rate:            15.8%         ║
║  Avg Win PnL:     $      7.46        ║
║  Avg Loss PnL:    $     -2.40        ║
║  Transaction Costs: $  777.70        ║
║  Avg Risk per Trade:$  1395.80       ║
║  Avg Position (BTC): 0.013705        ║
║  Avg Position ($):  $  1395.80       ║
╠══════════════════════════════════════╣
║ RISK METRICS                         ║
║  Sharpe Ratio:          3.03         ║
║  Max Drawdown: